# Cerebro-X — Real ADNI MRI CNN Training on Kaggle

This notebook trains `MRICerebroNet` only on a labelled, real ADNI T1w NIfTI dataset. It never creates synthetic scans or falls back to synthetic data.

> Research use only. ADNI data must be used in accordance with its data-use agreement and must not be made public as a Kaggle dataset.

## Before running

1. In Kaggle Notebook settings, enable a **GPU accelerator** and Internet if package installation is needed.
2. Add a private dataset containing this project ZIP (`cerebro_x_codebase.zip`).
3. Add your private ADNI dataset containing `manifest_with_labels.csv` and the NIfTI directory referenced by its `nifti_path` column.
4. Do not publish the notebook, its inputs, or outputs while they contain ADNI participant data.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/cerebro-x')
CODE_ZIPS = list(INPUT_ROOT.rglob('cerebro_x_codebase.zip')) + list(INPUT_ROOT.rglob('cerebro_x_code.zip'))
if not CODE_ZIPS:
    raise FileNotFoundError('Attach a private project dataset containing cerebro_x_codebase.zip.')

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIPS[0]) as archive:
    archive.extractall(WORK_ROOT)

# Some ZIPs contain a single top-level folder; locate the actual repository.
REPOS = [p.parent for p in WORK_ROOT.rglob('scripts/train_cnn3d.py')]
if len(REPOS) != 1:
    raise RuntimeError(f'Expected one project root, found: {REPOS}')
REPO = REPOS[0]
os.chdir(REPO)
print(f'Project root: {REPO}')

# Kaggle already supplies CUDA PyTorch. Install missing project packages without replacing it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel>=5.1.0', 'pandas>=2.0', 'scikit-learn>=1.3'], check=True)
sys.path.insert(0, str(REPO / 'src'))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No Kaggle GPU detected. Enable GPU in Notebook settings before training.')
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Discover ADNI inputs instead of hard-coding a Kaggle dataset slug.
MANIFESTS = list(INPUT_ROOT.rglob('manifest_with_labels.csv'))
if len(MANIFESTS) != 1:
    raise FileNotFoundError(f'Expected exactly one manifest_with_labels.csv, found: {MANIFESTS}')
MANIFEST = MANIFESTS[0]

import pandas as pd
manifest_df = pd.read_csv(MANIFEST)
required = {'subject_id', 'nifti_path', 'cdr_score'}
missing = required - set(manifest_df.columns)
if missing:
    raise ValueError(f'Manifest is missing required columns: {sorted(missing)}')

# The NIfTI base is normally the manifest's parent or a sibling folder.
candidates = [MANIFEST.parent, MANIFEST.parent / 'nifti']
NIFTI_ROOT = next((p for p in candidates if p.exists() and any(p.rglob('*.nii*'))), None)
if NIFTI_ROOT is None:
    raise FileNotFoundError('No .nii/.nii.gz files found beside the ADNI manifest. Check the private dataset layout.')

print('Manifest:', MANIFEST)
print('NIfTI root:', NIFTI_ROOT)
print('Scans:', len(manifest_df), '| Subjects:', manifest_df['subject_id'].nunique())
display(manifest_df[['subject_id', 'nifti_path', 'cdr_score']].head())

In [ ]:
# Validate one real scan and one training step before the full run.
SMOKE = [
    sys.executable, 'scripts/gpu_smoke_test.py',
    '--manifest', str(MANIFEST),
    '--nifti-dir', str(NIFTI_ROOT),
]
subprocess.run(SMOKE, check=True)

In [ ]:
# Full real-data training. Keep the default batch size of 4 for common 16 GB Kaggle GPUs.
OUTPUT_DIR = Path('/kaggle/working/EXP-MRI-ADNI-001')
TRAIN = [
    sys.executable, 'scripts/train_cnn3d.py',
    '--manifest', str(MANIFEST),
    '--nifti-dir', str(NIFTI_ROOT),
    '--out-dir', str(OUTPUT_DIR),
    '--epochs', '50', '--batch-size', '4', '--seed', '42', '--augment',
]
subprocess.run(TRAIN, check=True)

In [ ]:
# Export only trained artifacts and metrics; never package ADNI source data.
import json

print(json.dumps(json.loads((OUTPUT_DIR / 'metrics.json').read_text()), indent=2))
archive = shutil.make_archive('/kaggle/working/cerebro_x_mri_adni_artifact', 'zip', OUTPUT_DIR)
print(f'Download this output file: {archive}')

## After download

Extract the archive into this project as `artifacts/EXP-MRI-ADNI-001/`, then restart the API. The model loader will discover `best_checkpoint.pt` automatically. Do not copy the ADNI input data into the project or commit it to Git.